## DasyNet for patch-analysis

In [ ]:
# ── Step 1: Mount Drive and install ──────────────────────────
from google.colab import drive
drive.mount('/content/drive')

!pip install medmnist scikit-learn --quiet

import os
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset, ConcatDataset
import torchvision.transforms as transforms

import medmnist
from medmnist import INFO, PneumoniaMNIST
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, confusion_matrix, ConfusionMatrixDisplay
)
from PIL import Image

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
print('Libraries loaded!')

In [ ]:
# ── Step 2: Constants and paths ───────────────────────────────
DATA_FLAG  = 'pneumoniamnist'
IMG_SIZE   = 224
BATCH_SIZE = 64
DATA_ROOT  = os.path.expanduser('~/.medmnist')

CORRUPTED_DIR = '/content/drive/Shareddrives/Data_Corruption'
WEIGHTS_FILE  = '/content/drive/Shareddrives/Data_Corruption/dasynet_pneumonia.pth'
FT_WEIGHTS    = '/content/drive/Shareddrives/Data_Corruption/dasynet_finetuned_grid.pth'

FINETUNE_EPOCHS = 5
FINETUNE_LR     = 1e-4

ATTACK_NAMES = [
    'brightness_contrast',
    'jpeg_compression',
    'downsampling',
    'gaussian_blur'
]

info      = INFO[DATA_FLAG]
n_classes = len(info['label'])

print('Constants set ✓')

In [ ]:
# ── Step 3: DASYNet architecture ──────────────────────────────
class DASYNET(nn.Module):
    def __init__(self, in_channels=1, num_classes=2):
        super(DASYNET, self).__init__()
        self.block1 = nn.Sequential(
            nn.Conv2d(in_channels, 16, kernel_size=3, padding=1),
            nn.BatchNorm2d(16), nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2)
        )
        self.block2 = nn.Sequential(
            nn.Conv2d(16, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32), nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2)
        )
        self.block3 = nn.Sequential(
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64), nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2)
        )
        self.global_pool = nn.AdaptiveAvgPool2d((3, 3))
        self.flatten     = nn.Flatten()
        self.classifier  = nn.Sequential(
            nn.Linear(64 * 3 * 3, 128), nn.ReLU(), nn.Dropout(0.5),
            nn.Linear(128, num_classes)
        )

    def forward(self, x):
        x = self.block1(x)
        x = self.block2(x)
        x = self.block3(x)
        x = self.global_pool(x)
        x = self.flatten(x)
        return self.classifier(x)

model = DASYNET(in_channels=1, num_classes=2)
ckpt  = torch.load(WEIGHTS_FILE, map_location=device)
model.load_state_dict(ckpt)
model.to(device)
model.eval()
print(f'DASYNet loaded — {sum(p.numel() for p in model.parameters()):,} parameters ✓')

In [ ]:
# ── Step 4: Load clean train and val sets ─────────────────────
clean_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE), interpolation=Image.NEAREST),
    transforms.Lambda(lambda img: img.convert('L')),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5], std=[0.5])
])

train_dataset_clean = PneumoniaMNIST(split='train', transform=clean_transform,
                                      download=True, root=DATA_ROOT, size=IMG_SIZE)
val_dataset_clean   = PneumoniaMNIST(split='val',   transform=clean_transform,
                                      download=True, root=DATA_ROOT, size=IMG_SIZE)

def load_all(dataset):
    loader = DataLoader(dataset, batch_size=len(dataset), shuffle=False)
    images, labels = next(iter(loader))
    return images, labels

print('Loading clean train set...')
clean_train_images, clean_train_labels = load_all(train_dataset_clean)
print(f'  Clean train: {clean_train_images.shape}')

print('Loading clean val set...')
clean_val_images, clean_val_labels = load_all(val_dataset_clean)
print(f'  Clean val:   {clean_val_images.shape}')

In [ ]:
# ── Step 5: Load corrupted train and val sets ─────────────────
print('Loading corrupted train set...')
train_data = torch.load(os.path.join(CORRUPTED_DIR, 'train_corrupted.pt'))
corrupt_train_images = train_data['images']
corrupt_train_labels = train_data['labels'].squeeze()
print(f'  Corrupted train: {corrupt_train_images.shape}')
del train_data

print('Loading corrupted val set...')
val_data = torch.load(os.path.join(CORRUPTED_DIR, 'val_corrupted.pt'))
corrupt_val_images = val_data['images']
corrupt_val_labels = val_data['labels'].squeeze()
print(f'  Corrupted val:   {corrupt_val_images.shape}')
del val_data

In [ ]:
# ── Step 6: Combine clean + corrupted ────────────────────────
# Clean train + corrupted train
combined_train_images = torch.cat([clean_train_images, corrupt_train_images], dim=0)
combined_train_labels = torch.cat([clean_train_labels, corrupt_train_labels], dim=0)

# Clean val + corrupted val
combined_val_images = torch.cat([clean_val_images, corrupt_val_images], dim=0)
combined_val_labels = torch.cat([clean_val_labels, corrupt_val_labels], dim=0)

print(f'Combined train: {combined_train_images.shape[0]:,} images')
print(f'  Clean:        {clean_train_images.shape[0]:,}')
print(f'  Corrupted:    {corrupt_train_images.shape[0]:,}')
print(f'Combined val:   {combined_val_images.shape[0]:,} images')
print(f'  Clean:        {clean_val_images.shape[0]:,}')
print(f'  Corrupted:    {corrupt_val_images.shape[0]:,}')

# Free up memory
del clean_train_images, corrupt_train_images
del clean_val_images, corrupt_val_images
torch.cuda.empty_cache()

In [ ]:
# ── Step 7: Create DataLoaders ────────────────────────────────
train_dataset = TensorDataset(
    combined_train_images,
    combined_train_labels.long()
)
val_dataset = TensorDataset(
    combined_val_images,
    combined_val_labels.long()
)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE,
                          shuffle=True, pin_memory=True)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE,
                          shuffle=False, pin_memory=True)

print(f'Train loader: {len(train_loader)} batches')
print(f'Val loader:   {len(val_loader)} batches')

In [ ]:
# ── Step 8: Fine-tune ─────────────────────────────────────────
criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model.parameters(), lr=FINETUNE_LR, weight_decay=1e-4)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', factor=0.5, patience=2
)

history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}

print(f'Fine-tuning for {FINETUNE_EPOCHS} epochs...\n')

for epoch in range(FINETUNE_EPOCHS):

    # ── Train ─────────────────────────────────────────────────
    model.train()
    run_loss, correct, total = 0.0, 0, 0
    for imgs, lbls in train_loader:
        imgs, lbls = imgs.to(device), lbls.to(device)
        out  = model(imgs)
        loss = criterion(out, lbls)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        run_loss += loss.item() * imgs.size(0)
        correct  += out.argmax(1).eq(lbls).sum().item()
        total    += lbls.size(0)

    train_loss = run_loss / total
    train_acc  = 100. * correct / total

    # ── Validate ──────────────────────────────────────────────
    model.eval()
    val_loss, val_correct, val_total = 0.0, 0, 0
    with torch.no_grad():
        for imgs, lbls in val_loader:
            imgs, lbls = imgs.to(device), lbls.to(device)
            out  = model(imgs)
            loss = criterion(out, lbls)
            val_loss    += loss.item() * imgs.size(0)
            val_correct += out.argmax(1).eq(lbls).sum().item()
            val_total   += lbls.size(0)

    val_loss = val_loss / val_total
    val_acc  = 100. * val_correct / val_total

    history['train_loss'].append(train_loss)
    history['train_acc'].append(train_acc)
    history['val_loss'].append(val_loss)
    history['val_acc'].append(val_acc)
    scheduler.step(val_loss)

    print(f'Epoch [{epoch+1:02d}/{FINETUNE_EPOCHS}]  '
          f'Train Loss: {train_loss:.4f}  Train Acc: {train_acc:.2f}%  |  '
          f'Val Loss: {val_loss:.4f}  Val Acc: {val_acc:.2f}%')

# Save fine-tuned weights
torch.save(model.state_dict(), FT_WEIGHTS)
print(f'\n✅ Fine-tuned weights saved to: {FT_WEIGHTS}')

In [ ]:
# ── Step 9: Training curves ───────────────────────────────────
epochs = range(1, FINETUNE_EPOCHS + 1)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Loss
axes[0].plot(epochs, history['train_loss'], 'o-', color='steelblue', label='Train')
axes[0].plot(epochs, history['val_loss'],   's--', color='darkorange', label='Val')
axes[0].set_title('Fine-tuning Loss', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].legend()
axes[0].grid(alpha=0.3)

# Accuracy
axes[1].plot(epochs, history['train_acc'], 'o-', color='steelblue', label='Train')
axes[1].plot(epochs, history['val_acc'],   's--', color='darkorange', label='Val')
axes[1].set_title('Fine-tuning Accuracy', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy (%)')
axes[1].legend()
axes[1].grid(alpha=0.3)

fig.suptitle('DASYNet Fine-Tuning on Clean + Grid-Corrupted Data',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('dasynet_grid_finetune_curves.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Step 10: Evaluate on clean test set ──────────────────────
test_dataset_clean = PneumoniaMNIST(split='test', transform=clean_transform,
                                     download=True, root=DATA_ROOT, size=IMG_SIZE)
test_loader_clean  = DataLoader(test_dataset_clean, batch_size=BATCH_SIZE, shuffle=False)

model.eval()
preds, labels, probs = [], [], []
with torch.no_grad():
    for imgs, lbls in test_loader_clean:
        logits = model(imgs.to(device))
        p      = F.softmax(logits, dim=1)
        preds.extend(logits.argmax(1).cpu().numpy())
        labels.extend(lbls.squeeze().numpy())
        probs.extend(p[:, 1].cpu().numpy())

y_pred  = np.array(preds)
y_true  = np.array(labels)
y_probs = np.array(probs)

print('\n=== Fine-tuned DASYNet — Clean Test Set ===')
print(f'  Accuracy : {accuracy_score(y_true, y_pred):.4f}')
print(f'  Precision: {precision_score(y_true, y_pred, zero_division=0):.4f}')
print(f'  Recall   : {recall_score(y_true, y_pred, zero_division=0):.4f}')
print(f'  F1       : {f1_score(y_true, y_pred, zero_division=0):.4f}')
print(f'  AUROC    : {roc_auc_score(y_true, y_probs):.4f}')

# Confusion matrix
cm = confusion_matrix(y_true, y_pred)
fig, ax = plt.subplots(figsize=(5, 4))
ConfusionMatrixDisplay(cm, display_labels=['Normal', 'Pneumonia']).plot(
    ax=ax, colorbar=False, cmap='Blues'
)
ax.set_title('Fine-tuned DASYNet — Clean Test Set')
plt.tight_layout()
plt.savefig('dasynet_grid_finetuned_confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'TN={cm[0,0]}  FP={cm[0,1]}  FN={cm[1,0]}  TP={cm[1,1]}')